In [ ]:
import gzip
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.spatial.distance import correlation

train_attempt_count, worker_count, round_count, epoch_count, batch_count = 4, 2, 2, 30, 17

path_to_files = [f"exp_data/gradients_resnet/gradients_resnet_t{i}/" for i in range(0, 4)]

filename = f"_round_{0}_epoch_{0}_batch_{0}_gradients.pt.gz"
with gzip.open(path_to_files[0] + f"worker_{0}" + filename, "rb") as f:
    temp1 = torch.load(f)
    layer_counts = len(temp1)
    per_layer_ele_count = [len(t) for t in temp1]

In [ ]:
def get_corr(sample_list):
    return [[np.correlate(e[0], e[1]), 0] for e in sample_list]


all_data = []  # layer_id, time_(round, epoch), ele_id, corr_(corr, MI, dist, cos)
for layer_name_idx in range(layer_counts):
    layer_corr_over_t = []  # time_(round, epoch), ele_id, corr_(corr, MI, dist, cos)
    temp1 = np.array(np.meshgrid(range(round_count), range(epoch_count))).T.reshape(-1, 2)
    for curr_round, current_epoch in temp1:
        sample_list = [[], []]  # worker_id, sample_id_(ta_id, batch_idx), element_id

        temp2 = np.array(np.meshgrid(range(train_attempt_count), range(batch_count))).T.reshape(-1, 2)
        for train_attempt, batch_idx in temp2:
            filename = f"_round_{curr_round}_epoch_{current_epoch}_batch_{batch_idx}_gradients.pt.gz"

            with gzip.open(path_to_files[train_attempt] + f"worker_{0}" + filename, "rb") as f:
                grads_w_1 = torch.load(f)[layer_name_idx]
            with gzip.open(path_to_files[train_attempt] + f"worker_{1}" + filename, "rb") as f:
                grads_w_2 = torch.load(f)[layer_name_idx]

            sample_list[0].append(grads_w_1)
            sample_list[1].append(grads_w_2)
        sample_list = np.array(sample_list).transpose(2, 0, 1)
        sample_list = get_corr(sample_list)

        layer_corr_over_t.append(sample_list)
    layer_corr_over_t = np.array(layer_corr_over_t)

    all_data.append(layer_corr_over_t)
all_data = np.array(all_data)